# YOLOv8 fine-tune on Colab (Drive dataset)

사용 순서:
1. **런타임 → 런타임 유형 변경 → GPU (T4/L4 권장)**
2. 아래 `CONFIG` 셀에서 Drive 내 데이터셋 경로(`DATASET_DIR`)만 본인 것으로 바꿔서 실행.
3. 끝나면 `best.pt`가 Drive의 `OUTPUT_DIR` 아래에 자동 저장됨. WSL2에서 그걸 받아서 inference에 사용.

## 0. Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. CONFIG — 여기만 손대면 됨

- `DATASET_DIR`: Roboflow 다운로드 폴더 (그 안에 `data.yaml`, `train/`, `valid/`, (선택)`test/`).
- `OUTPUT_DIR`: 학습 결과(`best.pt`, `last.pt`, 메트릭)가 들어갈 Drive 폴더. 자동 생성됨.
- 모델 크기는 `yolov8n.pt`가 기본. 더 키우려면 `yolov8s.pt` / `yolov8m.pt`.

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/AUE4040/roboflow_dataset'   # ← 본인 경로로 변경
OUTPUT_DIR  = '/content/drive/MyDrive/AUE4040/yolo_runs'

MODEL    = 'yolov8n.pt'   # n / s / m / l / x
EPOCHS   = 100
IMGSZ    = 320            # Jetson 배포 목표 입력 크기와 맞춤 (PROJECT_PLAN §4-A)
BATCH    = 32             # T4 16GB 기준. OOM 나면 16으로
PATIENCE = 30             # early-stop
RUN_NAME = 'rover_v1'

import os
DATA_YAML = os.path.join(DATASET_DIR, 'data.yaml')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('data.yaml :', DATA_YAML, 'exists=', os.path.exists(DATA_YAML))
print('output    :', OUTPUT_DIR)

## 2. 데이터셋 sanity check

`data.yaml`의 클래스 names와 경로가 Drive 마운트 경로 기준으로 맞는지 확인. Roboflow가 만든 `data.yaml`은 보통 상대경로(`../train/images`)라서 그대로 잘 동작하지만, 절대경로로 박혀있으면 여기서 보여줌.

In [ ]:
import yaml, pathlib
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print('nc   :', cfg.get('nc'))
print('names:', cfg.get('names'))
for k in ('train','val','test'):
    if k in cfg:
        p = cfg[k]
        ap = (pathlib.Path(DATASET_DIR) / p).resolve() if not os.path.isabs(p) else pathlib.Path(p)
        print(f'{k:5s}: {p}  →  {ap}  exists={ap.exists()}')

## 3. ultralytics 설치

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. 학습

결과는 `OUTPUT_DIR/RUN_NAME/weights/best.pt`에 저장.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    # augmentation: 폐쇄 환경 + 표지판/신호등이라 mosaic는 약하게
    mosaic=0.5,
    mixup=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
)

## 5. 검증 (best.pt로 val 셋)

In [ ]:
best_pt = os.path.join(OUTPUT_DIR, RUN_NAME, 'weights', 'best.pt')
print('best.pt:', best_pt, 'exists=', os.path.exists(best_pt))

best = YOLO(best_pt)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split='val')
print('mAP50  :', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))
print('names  :', best.names)

## 6. 샘플 추론 (val 셋 일부 시각화)

In [ ]:
import glob
from IPython.display import Image, display

val_dir = None
for cand in (cfg.get('val'), 'valid/images', 'val/images'):
    if not cand:
        continue
    p = cand if os.path.isabs(cand) else os.path.join(DATASET_DIR, cand)
    if os.path.isdir(p):
        val_dir = p; break
print('val dir:', val_dir)

if val_dir:
    sample = sorted(glob.glob(os.path.join(val_dir, '*')))[:6]
    preds = best.predict(sample, imgsz=IMGSZ, conf=0.25, save=True,
                         project=OUTPUT_DIR, name=RUN_NAME + '_preview', exist_ok=True)
    for p in preds:
        display(Image(filename=p.save_dir + '/' + os.path.basename(p.path)))

## 7. (선택) ONNX export

Jetson에 TRT engine으로 올릴 거면 ONNX가 중간 산출물. WSL2 standalone inference만이라면 건너뛰어도 됨.

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print('ONNX saved next to best.pt:', os.path.join(OUTPUT_DIR, RUN_NAME, 'weights'))

## 8. 끝났으면 Drive에서 best.pt 가져가기

WSL2에서:
```
# Drive 데스크탑 동기화 폴더에서 복사
cp '<Drive sync 경로>/AUE4040/yolo_runs/rover_v1/weights/best.pt' \\
   ~/Personal_Research/AUE4040/main/best.pt
```

그 다음 클래스 names를 확인하고 `params.yaml` / `decision_node.py`의 rule-base 라벨을 맞춰주면 됨.